In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import torch
import copy
import itertools

from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.graphs import StructureGraph
from pymatgen.analysis import local_env

from networkx.algorithms.components import is_connected

from sklearn.metrics import accuracy_score, recall_score, precision_score

from torch_scatter import scatter

from p_tqdm import p_umap

import ast
#import the random function library
import random

import os 

from tqdm.auto import tqdm
tqdm.pandas()

train_df = pd.read_csv("/home/gridsan/tmackey/cdvae/data/mp_20/train.csv")

In [2]:

CrystalNN = local_env.CrystalNN(
    distance_cutoffs=None, x_diff_weight=-1, porous_adjustment=False)

In [9]:
# Get the job array index from the environment variable
# job_index = int(os.getenv('SLURM_ARRAY_TASK_ID', 0))
# total_tasks = int(os.getenv('SLURM_ARRAY_TASK_COUNT', 1))

job_index = 0
total_tasks = 20

# Split the dataset into chunks for processing
total_structures = len(train_df)
chunk_size = total_structures // total_tasks # this rounds down
start_index = job_index * chunk_size
end_index = min(start_index + chunk_size, total_structures)

In [4]:


def build_crystal(crystal_str, niggli=True, primitive=False):
    """Build crystal from cif string."""
    crystal = Structure.from_str(crystal_str, fmt='cif')

    if primitive:
        crystal = crystal.get_primitive_structure()

    if niggli:
        crystal = crystal.get_reduced_structure()

    canonical_crystal = Structure(
        lattice=Lattice.from_parameters(*crystal.lattice.parameters),
        species=crystal.species,
        coords=crystal.frac_coords,
        coords_are_cartesian=False,
    )
    # match is gaurantteed because cif only uses lattice params & frac_coords
    # assert canonical_crystal.matches(crystal)
    return canonical_crystal

In [12]:
33605 + 546

34151

In [22]:
job_index = 13
total_tasks = 21

#job_index = 0
#total_tasks = 20

# Split the dataset into chunks for processing
total_structures = len(train_df)
#divde the total number of structures by the number of tasks and round up
chunk_size = int(np.ceil(total_structures / total_tasks))
start_index = job_index * chunk_size
end_index = min(start_index + chunk_size, total_structures)

print(f"Processing chunk {job_index} of {total_tasks} with {chunk_size} structures")
print(f"Processing structures {start_index} to {end_index}")


Processing chunk 13 of 21 with 2585 structures
Processing structures 33605 to 36190


In [5]:
structures = train_df['cif'].progress_apply(build_crystal)


  0%|          | 0/54272 [00:00<?, ?it/s]

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/core/periodic_table.py:221: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(


In [ ]:
num_sites = structures.apply(lambda x: x.reduced_formula)

: 

In [27]:
#find which indexes have num_sites = 9
nine_sites = np.where(num_sites == 9)[0]

In [28]:
nine_sites

array([  11,   32,   35,   48,   51,   70,   72,   74,   86,  107,  108,
        121,  122,  151,  181,  202,  209,  258,  296,  353,  378,  386,
        392,  404,  407,  413,  429,  433,  441,  487,  498,  537,  606,
        612,  649,  651,  671,  683,  702,  703,  710,  717,  740,  766,
        785,  809,  821,  853,  858,  867,  882,  894,  896, 1035, 1036,
       1048, 1065, 1100, 1122, 1137, 1155, 1188, 1205, 1230, 1232, 1260,
       1262, 1274, 1289, 1301, 1340, 1351, 1361, 1376, 1396, 1408, 1413,
       1416, 1429, 1430, 1439, 1440, 1442, 1457, 1465, 1469, 1472, 1482,
       1483, 1484, 1506, 1521, 1555, 1597, 1634, 1636, 1637, 1638, 1675,
       1690, 1704, 1731, 1735, 1738, 1747, 1767, 1789, 1790, 1798, 1824,
       1834, 1838, 1845, 1862, 1872, 1880, 1935, 1943, 1959, 1961, 1992,
       2012, 2046, 2084, 2129, 2150, 2158, 2206, 2209, 2223, 2233, 2271,
       2313, 2370, 2389, 2403, 2412, 2415, 2435, 2454, 2467, 2475, 2516,
       2520, 2562, 2569])

In [31]:


result_graphs = structures[600:606].progress_apply(lambda x: StructureGraph.with_local_env_strategy(x, CrystalNN))




  0%|          | 0/6 [00:00<?, ?it/s]

In [30]:
result_graphs

34141    Structure Graph\nStructure: \nFull Formula (Ca...
34142    Structure Graph\nStructure: \nFull Formula (Zr...
34143    Structure Graph\nStructure: \nFull Formula (Tb...
34144    Structure Graph\nStructure: \nFull Formula (Ho...
Name: cif, dtype: object